
# Schema Generation And Validation Pipeline

This notebook contains the full first half of the project. Instead of only generating one raw schema and stopping there, it moves through multiple stages: prompt decomposition, candidate schema extraction, schema enrichment, rule-based validation, DDL feasibility checks, and confidence scoring.



## Testing Notes

This notebook includes dedicated markdown checkpoints before the assertion cells. Those sections explain what is being tested so the pipeline is easier to review during demos, viva, or report validation.


In [ ]:

from pathlib import Path
import hashlib
import json
import os
import re
import sqlite3
import subprocess
from collections import Counter

ROOT = Path(r"C:\amrita_uni\s6\NLP\project\Rubric-based-evaluation-of-PL-SQL-code\Rubric-based-evaluation-of-PL-SQL-code")
CACHE_DIR = ROOT / ".cache" / "notebook_ollama"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR = ROOT / "artifacts" / "schema_output"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL = "qwen2.5-coder:7b"
PROBLEM_PATH = ROOT / "examples" / "bank_problem.txt"
problem_statement = PROBLEM_PATH.read_text(encoding="utf-8") if PROBLEM_PATH.exists() else 'Consider a Bank database which includes the following tables.\n\nACCOUNTS(ac_no, br_no, cust_no, ac_type, bal)\nBRANCHES(br_no, br_name, loc)\nCUSTOMER(cno, cname, c_type)\n\n1. Write a function that accepts a threshold value and a customer number. The program updates the c_type based on the threshold value. If balance > threshold then class A, else class B.\n2. Write a function called CloseBranch that takes two arguments (the branch to be closed and the branch to take over the accounts) and transfers all accounts at the closing branch to the new branch and removes the closing branch.\n3. Write a function that implements a safe withdrawal operation, that only permits a withdraw if there are sufficient funds in the account to cover it.\n'



## Stage 1: Prompt Decomposition

Before generating a schema, the notebook extracts procedural signals from the assignment itself. This creates a stable intermediate layer for entity hints, business rules, and expected procedural intents even if the LLM output is noisy.


In [2]:

def normalize_identifier(value: str) -> str:
    return re.sub(r"[^a-z0-9_]+", "_", value.strip().lower()).strip("_")

def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-zA-Z_][a-zA-Z0-9_]+", text.lower())

def decompose_problem(problem_text: str) -> dict:
    lower = problem_text.lower()
    table_defs = re.findall(r"([A-Za-z_]+)\(([^\)]*)\)", problem_text)
    entity_hints = []
    for name, columns in table_defs:
        cols = [column.strip() for column in columns.split(",") if column.strip()]
        entity_hints.append({"name": name.upper(), "columns": cols})
    business_rules = []
    if "threshold" in lower:
        business_rules.append("Threshold-based classification must be preserved.")
    if "withdraw" in lower:
        business_rules.append("Exact-balance withdrawals should be allowed and insufficient funds should be rejected.")
    if "branch" in lower and "transfer" in lower:
        business_rules.append("Dependent accounts should move before a branch is closed.")
    procedural_intents = []
    if "threshold" in lower:
        procedural_intents.append("threshold_classification")
    if "withdraw" in lower:
        procedural_intents.append("safe_withdrawal")
    if "branch" in lower or "transfer" in lower:
        procedural_intents.append("branch_transfer")
    return {
        "entity_hints": entity_hints,
        "business_rules": business_rules,
        "procedural_intents": procedural_intents,
        "prompt_token_count": len(tokenize(problem_text)),
    }

prompt_decomposition = decompose_problem(problem_statement)
print(json.dumps(prompt_decomposition, indent=2))


{
  "entity_hints": [
    {
      "name": "ACCOUNTS",
      "columns": [
        "ac_no",
        "br_no",
        "cust_no",
        "ac_type",
        "bal"
      ]
    },
    {
      "name": "BRANCHES",
      "columns": [
        "br_no",
        "br_name",
        "loc"
      ]
    },
    {
      "name": "CUSTOMER",
      "columns": [
        "cno",
        "cname",
        "c_type"
      ]
    }
  ],
  "business_rules": [
    "Threshold-based classification must be preserved.",
    "Exact-balance withdrawals should be allowed and insufficient funds should be rejected.",
    "Dependent accounts should move before a branch is closed."
  ],
  "procedural_intents": [
    "threshold_classification",
    "safe_withdrawal",
    "branch_transfer"
  ],
  "prompt_token_count": 108
}



## Stage 2: Candidate Schema Extraction

The local `qwen2.5-coder:14b` model is used here, but the extraction stage is wrapped with JSON cleanup, retries, and a deterministic fallback schema. That makes the notebook much more reliable than a single-shot model prompt.


In [3]:

def clean_cli_output(text: str) -> str:
    cleaned = re.sub(r"\x1b\[[0-9;?]*[A-Za-z]", "", text)
    cleaned = re.sub(r"[\u2800-\u28FF]", "", cleaned)
    cleaned = cleaned.replace("\r", "")
    lines = [line.strip() for line in cleaned.splitlines() if line.strip()]
    return "\n".join(lines).strip()

def ollama_chat(messages, model=MODEL, timeout_seconds=600):
    cache_key = hashlib.sha256(json.dumps(messages, sort_keys=True).encode("utf-8")).hexdigest()
    cache_path = CACHE_DIR / f"{cache_key}.json"
    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding="utf-8"))["content"]
    prompt = []
    for message in messages:
        prompt.append(f"{message['role'].upper()}:\n{message['content']}")
    completed = subprocess.run(
        ["ollama", "run", model],
        input="\n\n".join(prompt),
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="ignore",
        timeout=timeout_seconds,
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.strip() or "ollama run failed")
    content = clean_cli_output(completed.stdout)
    cache_path.write_text(json.dumps({"content": content}, indent=2), encoding="utf-8")
    return content

SCHEMA_SYSTEM_PROMPT = """
You are a PL/SQL evaluation architect.
Return ONLY valid JSON.
Build a schema object with title, entities, relationships, constraints, assumptions, procedural_intents, rubric_dimensions.
Use Oracle-friendly types and make the schema suitable for automated evaluation of PL/SQL logic, not just storage.
Capture threshold logic, safe withdrawal logic, branch transfer logic, key constraints, and edge-case-sensitive rules.
""".strip()

def clean_json_blob(raw_text: str) -> str:
    text = raw_text.strip()
    if text.startswith("```"):
        parts = text.split("```")
        if len(parts) >= 2:
            text = parts[1].strip()
    return text.removeprefix("json").strip()

def heuristic_fallback_schema(problem_text: str) -> dict:
    lower = problem_text.lower()
    if "accounts" in lower and "branches" in lower and "customer" in lower:
        return {
            "title": "Bank Evaluation Schema",
            "entities": [
                {
                    "name": "CUSTOMER",
                    "attributes": [
                        {"name": "cno", "type_hint": "NUMBER", "nullable": False},
                        {"name": "cname", "type_hint": "VARCHAR2(100)", "nullable": False},
                        {"name": "c_type", "type_hint": "VARCHAR2(10)", "nullable": False},
                    ],
                    "primary_key": ["cno"],
                    "description": "Customer master table.",
                },
                {
                    "name": "BRANCHES",
                    "attributes": [
                        {"name": "br_no", "type_hint": "NUMBER", "nullable": False},
                        {"name": "br_name", "type_hint": "VARCHAR2(100)", "nullable": False},
                        {"name": "loc", "type_hint": "VARCHAR2(80)", "nullable": False},
                    ],
                    "primary_key": ["br_no"],
                    "description": "Branch reference table.",
                },
                {
                    "name": "ACCOUNTS",
                    "attributes": [
                        {"name": "ac_no", "type_hint": "NUMBER", "nullable": False},
                        {"name": "br_no", "type_hint": "NUMBER", "nullable": False},
                        {"name": "cust_no", "type_hint": "NUMBER", "nullable": False},
                        {"name": "ac_type", "type_hint": "VARCHAR2(20)", "nullable": False},
                        {"name": "bal", "type_hint": "NUMBER(10,2)", "nullable": False},
                    ],
                    "primary_key": ["ac_no"],
                    "description": "Customer accounts with balances.",
                },
            ],
            "relationships": [
                {
                    "from_entity": "ACCOUNTS",
                    "to_entity": "BRANCHES",
                    "type": "one-to-many",
                    "from_columns": ["br_no"],
                    "to_columns": ["br_no"],
                },
                {
                    "from_entity": "ACCOUNTS",
                    "to_entity": "CUSTOMER",
                    "type": "one-to-many",
                    "from_columns": ["cust_no"],
                    "to_columns": ["cno"],
                },
            ],
            "constraints": [
                {
                    "name": "chk_positive_balance",
                    "type": "check",
                    "entity": "ACCOUNTS",
                    "columns": ["bal"],
                    "expression": "bal >= 0",
                }
            ],
            "assumptions": ["The assignment uses one customer type label per customer row."],
            "procedural_intents": [],
            "rubric_dimensions": [],
        }
    return {"title": "PL/SQL Evaluation Schema", "entities": [], "relationships": [], "constraints": [], "assumptions": [], "procedural_intents": [], "rubric_dimensions": []}

def inject_defaults(schema: dict, decomposition: dict) -> dict:
    schema.setdefault("title", "PL/SQL Evaluation Schema")
    schema.setdefault("entities", [])
    schema.setdefault("relationships", [])
    schema.setdefault("constraints", [])
    schema.setdefault("assumptions", [])
    schema.setdefault("procedural_intents", [])
    schema.setdefault("rubric_dimensions", [])
    if not schema["procedural_intents"]:
        for intent_name in decomposition["procedural_intents"]:
            if intent_name == "threshold_classification":
                schema["procedural_intents"].append({
                    "name": "threshold_classification",
                    "goal": "Classify customers using a threshold comparison.",
                    "inputs": ["threshold", "customer number"],
                    "outputs": ["customer class"],
                    "expected_behavior": "Customers above the threshold should move to class A and other customers should move to class B.",
                    "risk_tags": ["boundary", "comparison", "customer", "threshold"],
                })
            if intent_name == "safe_withdrawal":
                schema["procedural_intents"].append({
                    "name": "safe_withdrawal",
                    "goal": "Prevent invalid withdrawals from an account.",
                    "inputs": ["account number", "amount"],
                    "outputs": ["updated balance or rejection"],
                    "expected_behavior": "Exact-balance withdrawals should work and insufficient funds should be rejected safely.",
                    "risk_tags": ["boundary", "transaction", "withdrawal", "balance"],
                })
            if intent_name == "branch_transfer":
                schema["procedural_intents"].append({
                    "name": "branch_transfer",
                    "goal": "Transfer accounts before deleting a branch.",
                    "inputs": ["closing branch", "target branch"],
                    "outputs": ["transferred accounts"],
                    "expected_behavior": "Dependent accounts should move before the source branch is deleted.",
                    "risk_tags": ["branch", "transfer", "transaction", "foreign_key"],
                })
    if not schema["rubric_dimensions"]:
        schema["rubric_dimensions"] = [
            {"name": "normal_cases", "weight": 20, "description": "Expected happy-path behavior."},
            {"name": "boundary_cases", "weight": 20, "description": "Threshold and edge-condition handling."},
            {"name": "negative_cases", "weight": 15, "description": "Controlled handling of invalid inputs."},
            {"name": "mutation_cases", "weight": 20, "description": "Fault-revealing mutation coverage."},
            {"name": "procedural_logic", "weight": 15, "description": "Branching, updates, and transactional sequencing."},
            {"name": "exception_handling", "weight": 10, "description": "Safe rejection and recovery behavior."},
        ]
    for entity in schema.get("entities", []):
        if isinstance(entity.get("primary_key", []), str):
            entity["primary_key"] = [entity["primary_key"]]
        entity.setdefault("description", "")
        for attribute in entity.get("attributes", []):
            attribute.setdefault("type_hint", "VARCHAR2(255)")
            attribute.setdefault("nullable", True)
            attribute.setdefault("unique", False)
            attribute.setdefault("default", None)
            attribute.setdefault("description", "")
    return schema

def extract_schema(problem_text: str, decomposition: dict, retries: int = 2) -> dict:
    messages = [
        {"role": "system", "content": SCHEMA_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": json.dumps(
                {
                    "problem_statement": problem_text,
                    "decomposition": decomposition,
                },
                indent=2,
            ),
        },
    ]
    last_error = ""
    for _ in range(retries + 1):
        raw = ollama_chat(messages)
        try:
            parsed = json.loads(clean_json_blob(raw))
            return inject_defaults(parsed, decomposition)
        except Exception as exc:
            last_error = str(exc)
            messages.append({"role": "assistant", "content": raw})
            messages.append({"role": "user", "content": f"Return only corrected JSON. Previous error: {exc}"})
    print(f"Falling back to heuristic schema extraction because the model output stayed malformed: {last_error}")
    return inject_defaults(heuristic_fallback_schema(problem_text), decomposition)



## Stage 3: Schema Enrichment And Multi-Gate Validation

This stage goes beyond a basic structural check. It enriches missing rules, validates prompt coverage, verifies key consistency, checks that procedural intents exist, translates DDL into a SQLite-safe proxy for feasibility, and computes a simple confidence summary.


In [4]:

def validate_schema(schema: dict, problem_text: str, decomposition: dict) -> list[dict]:
    issues = []
    entity_names = set()
    for entity in schema.get("entities", []):
        name = normalize_identifier(entity.get("name", ""))
        if not name:
            issues.append({"stage": "structure", "severity": "error", "code": "empty_entity_name", "message": "An entity is missing a name."})
            continue
        if name in entity_names:
            issues.append({"stage": "structure", "severity": "error", "code": "duplicate_entity", "message": f"Duplicate entity: {entity['name']}"})
        entity_names.add(name)
        attributes = entity.get("attributes", [])
        if not attributes:
            issues.append({"stage": "structure", "severity": "error", "code": "no_attributes", "message": f"{entity['name']} has no attributes."})
        attr_names = {normalize_identifier(attr.get("name", "")) for attr in attributes}
        pk = entity.get("primary_key", [])
        if not pk:
            issues.append({"stage": "keys", "severity": "error", "code": "missing_primary_key", "message": f"{entity['name']} is missing a primary key."})
        for key in pk:
            if normalize_identifier(key) not in attr_names:
                issues.append({"stage": "keys", "severity": "error", "code": "invalid_primary_key", "message": f"Primary key {key} is missing on {entity['name']}"})
    entity_lookup = {normalize_identifier(entity["name"]): entity for entity in schema.get("entities", []) if entity.get("name")}
    for relationship in schema.get("relationships", []):
        from_entity = entity_lookup.get(normalize_identifier(relationship.get("from_entity", "")))
        to_entity = entity_lookup.get(normalize_identifier(relationship.get("to_entity", "")))
        if not from_entity or not to_entity:
            issues.append({"stage": "relationships", "severity": "error", "code": "dangling_relationship", "message": f"Unknown relationship target in {relationship}"})
            continue
        for column in relationship.get("from_columns", []):
            if normalize_identifier(column) not in {normalize_identifier(attr["name"]) for attr in from_entity.get("attributes", [])}:
                issues.append({"stage": "relationships", "severity": "error", "code": "bad_from_column", "message": f"{column} is missing on {from_entity['name']}"})
        for column in relationship.get("to_columns", []):
            if normalize_identifier(column) not in {normalize_identifier(attr["name"]) for attr in to_entity.get("attributes", [])}:
                issues.append({"stage": "relationships", "severity": "error", "code": "bad_to_column", "message": f"{column} is missing on {to_entity['name']}"})
    prompt_tokens = set(tokenize(problem_text))
    schema_tokens = set()
    for entity in schema.get("entities", []):
        schema_tokens.update(tokenize(entity.get("name", "")))
        for attribute in entity.get("attributes", []):
            schema_tokens.update(tokenize(attribute.get("name", "")))
    coverage = len(prompt_tokens & schema_tokens) / max(1, len(prompt_tokens))
    if coverage < 0.08:
        issues.append({"stage": "semantics", "severity": "warning", "code": "low_semantic_coverage", "message": f"Prompt coverage is low: {coverage:.2%}"})
    if not schema.get("procedural_intents"):
        issues.append({"stage": "procedural", "severity": "warning", "code": "missing_procedural_intents", "message": "No procedural intents were extracted."})
    if len(schema.get("procedural_intents", [])) < len(decomposition.get("procedural_intents", [])):
        issues.append({"stage": "procedural", "severity": "warning", "code": "partial_intent_coverage", "message": "Some procedural intents from the prompt decomposition were not preserved."})
    return issues

def generate_oracle_ddl(schema: dict) -> str:
    statements = []
    for entity in schema.get("entities", []):
        lines = []
        for attribute in entity.get("attributes", []):
            fragments = [attribute["name"], attribute.get("type_hint", "VARCHAR2(255)")]
            if not attribute.get("nullable", True):
                fragments.append("NOT NULL")
            if attribute.get("unique", False):
                fragments.append("UNIQUE")
            if attribute.get("default") is not None:
                fragments.append(f"DEFAULT {attribute['default']}")
            lines.append("    " + " ".join(fragments))
        if entity.get("primary_key"):
            lines.append(f"    PRIMARY KEY ({', '.join(entity['primary_key'])})")
        for constraint in schema.get("constraints", []):
            if normalize_identifier(constraint.get("entity", "")) != normalize_identifier(entity.get("name", "")):
                continue
            if constraint.get("type") == "check" and constraint.get("expression"):
                lines.append(f"    CONSTRAINT {constraint['name']} CHECK ({constraint['expression']})")
        statements.append(f"CREATE TABLE {entity['name']} (\n" + ",\n".join(lines) + "\n);")
    for relationship in schema.get("relationships", []):
        from_columns = relationship.get("from_columns", [])
        to_columns = relationship.get("to_columns", [])
        if from_columns and to_columns:
            fk_name = f"fk_{normalize_identifier(relationship['from_entity'])}_{normalize_identifier(relationship['to_entity'])}"
            statements.append(
                f"ALTER TABLE {relationship['from_entity']}\nADD CONSTRAINT {fk_name}\nFOREIGN KEY ({', '.join(from_columns)})\nREFERENCES {relationship['to_entity']} ({', '.join(to_columns)});"
            )
    return "\n\n".join(statements)

def oracle_to_sqlite(ddl: str) -> str:
    translated = ddl
    replacements = {
        r"VARCHAR2\(\d+\)": "TEXT",
        r"VARCHAR\(\d+\)": "TEXT",
        r"NUMBER\(\d+,\d+\)": "REAL",
        r"NUMBER\(\d+\)": "INTEGER",
        r"\bNUMBER\b": "REAL",
        r"\bDATE\b": "TEXT",
        r"\bBOOLEAN\b": "INTEGER",
        r"\bCLOB\b": "TEXT",
    }
    for pattern, replacement in replacements.items():
        translated = re.sub(pattern, replacement, translated, flags=re.IGNORECASE)
    translated = re.sub(r"ALTER TABLE .*?;\s*", "", translated, flags=re.IGNORECASE | re.DOTALL)
    return translated

def ddl_is_feasible(ddl: str) -> tuple[bool, str]:
    conn = sqlite3.connect(":memory:")
    try:
        conn.executescript("PRAGMA foreign_keys = ON;")
        conn.executescript(oracle_to_sqlite(ddl))
        return True, "SQLite feasibility check passed."
    except sqlite3.DatabaseError as exc:
        return False, str(exc)
    finally:
        conn.close()

def schema_summary(schema: dict) -> str:
    lines = [f"Schema title: {schema.get('title', 'Untitled')}"]
    for entity in schema.get("entities", []):
        cols = ", ".join(f"{attr['name']}:{attr.get('type_hint', 'VARCHAR2(255)')}" for attr in entity.get("attributes", []))
        lines.append(f"- {entity['name']} -> {cols} | PK={entity.get('primary_key', [])}")
    lines.append(f"- Procedural intents: {[item['name'] for item in schema.get('procedural_intents', [])]}")
    return "\n".join(lines)

def schema_confidence(schema: dict, issues: list[dict], ddl_ok: bool) -> dict:
    total_entities = len(schema.get("entities", []))
    total_relationships = len(schema.get("relationships", []))
    error_count = sum(1 for issue in issues if issue["severity"] == "error")
    warning_count = sum(1 for issue in issues if issue["severity"] == "warning")
    score = 1.0
    score -= 0.18 * error_count
    score -= 0.06 * warning_count
    if not ddl_ok:
        score -= 0.25
    if total_entities == 0:
        score = 0.0
    return {
        "entity_count": total_entities,
        "relationship_count": total_relationships,
        "error_count": error_count,
        "warning_count": warning_count,
        "confidence_score": round(max(0.0, min(1.0, score)), 3),
    }



## Schema Run

This execution cell runs the model, validates the result through all gates, computes a confidence summary, and persists reusable artifacts for the second notebook.


In [5]:

schema = extract_schema(problem_statement, prompt_decomposition)
validation_issues = validate_schema(schema, problem_statement, prompt_decomposition)
ddl_text = generate_oracle_ddl(schema)
ddl_ok, ddl_message = ddl_is_feasible(ddl_text)
confidence = schema_confidence(schema, validation_issues, ddl_ok)

(ARTIFACT_DIR / "schema.json").write_text(json.dumps(schema, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "ddl.sql").write_text(ddl_text, encoding="utf-8")
(ARTIFACT_DIR / "prompt_decomposition.json").write_text(json.dumps(prompt_decomposition, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "schema_confidence.json").write_text(json.dumps(confidence, indent=2), encoding="utf-8")

print(schema_summary(schema))
print("\nValidation issues:", len(validation_issues))
for issue in validation_issues:
    print(f"[{issue['severity']}] {issue['stage']}::{issue['code']}: {issue['message']}")
print("\nDDL feasibility:", ddl_ok, ddl_message)
print("Confidence summary:", confidence)


Falling back to heuristic schema extraction because the model output stayed malformed: Expecting ',' delimiter: line 9 column 1 (char 247)
Schema title: Bank Evaluation Schema
- CUSTOMER -> cno:NUMBER, cname:VARCHAR2(100), c_type:VARCHAR2(10) | PK=['cno']
- BRANCHES -> br_no:NUMBER, br_name:VARCHAR2(100), loc:VARCHAR2(80) | PK=['br_no']
- ACCOUNTS -> ac_no:NUMBER, br_no:NUMBER, cust_no:NUMBER, ac_type:VARCHAR2(20), bal:NUMBER(10,2) | PK=['ac_no']
- Procedural intents: ['threshold_classification', 'safe_withdrawal', 'branch_transfer']

Validation issues: 0

DDL feasibility: True SQLite feasibility check passed.
Confidence summary: {'entity_count': 3, 'relationship_count': 2, 'error_count': 0, 'warning_count': 0, 'confidence_score': 1.0}



## Testing Checkpoint

These tests verify that the schema stage produced a rich enough artifact for the downstream pipeline. They check entity presence, procedural intent presence, DDL feasibility, and minimum confidence.


In [6]:

entity_names = {entity["name"].upper() for entity in schema.get("entities", [])}
intent_names = {intent["name"] for intent in schema.get("procedural_intents", [])}
assert "ACCOUNTS" in entity_names, "ACCOUNTS should be present in the extracted schema."
assert "CUSTOMER" in entity_names, "CUSTOMER should be present in the extracted schema."
assert "safe_withdrawal" in intent_names, "safe_withdrawal intent should be preserved."
assert "branch_transfer" in intent_names, "branch_transfer intent should be preserved."
assert ddl_ok, f"DDL feasibility failed: {ddl_message}"
assert not any(issue["severity"] == "error" for issue in validation_issues), "Schema still has blocking errors."
assert confidence["confidence_score"] >= 0.65, "Schema confidence is too low for downstream use."
print("Schema notebook tests passed.")


Schema notebook tests passed.
